# Extraction des caractéristiques (Feature Extraction)
### (Fusion des niveaux 1 et 2, appliquée en boucle à chaque sujet présent dans `data/`)

Ce notebook reprend exactement la même logique déjà validée sur le sujet 1 (règle d'exclusion des essais liés au SOA, décodage des événements EEG, caractéristiques spectrales et FRN, conversion mBLL en HbO/HbR), mais l'applique automatiquement à tous les sujets disposant d'un triplet complet de fichiers `subN.mat` + `subN_tsk.mat` + `subN.nirs` dans `../data/`.

Chaque sujet est traité indépendamment, dans un bloc try/except : si un sujet pose problème (fichier manquant, données corrompues, format inattendu), il est simplement journalisé puis ignoré, sans interrompre le traitement des autres sujets.

Résultat produit : `../data/level2_features_all_subjects.csv`, un fichier avec une ligne par essai, identifiée par la colonne `subject_id`.

In [1]:
import os
import re
import glob

import h5py
import mat73
import numpy as np
import pandas as pd
from scipy.signal import welch
from scipy.stats import entropy as scipy_entropy

DATA_DIR = os.environ.get("EEG_FNIRS_DATA_DIR", "data")  # folder with subN.mat, subN_tsk.mat, subN.nirs

N_BLOCKS = 6
N_TRIALS_PER_BLOCK = 172
EXCLUDE_TRIAL_ROWS = [0, 43]  # standing exclusion rule, found on subject 1

BAND_THETA = (4, 7)
BAND_ALPHA = (8, 12)
BAND_BETA = (13, 30)
FRONTAL_CHANS = ['FZ', 'FCZ', 'FC3', 'FC4', 'F3', 'F4']
CHAN_NAMES6
EXT_COEFFS = {
    695: {'HbO': 1102.2, 'HbR': 3213.5},
    830: {'HbO': 2526.4, 'HbR': 1798.6},
}
DPF = 6.0
L = 3.0

## 1. Repérage des sujets disposant d'un triplet de fichiers complet

In [2]:
def discover_subjects(data_dir):
    beh_files = glob.glob(os.path.join(data_dir, "sub*.mat"))
    subject_ids = []
    for f in beh_files:
        name = os.path.basename(f)
        m = re.match(r"^sub(\d+)\.mat$", name)  # excludes subN_tsk.mat
        if m:
            subject_ids.append(int(m.group(1)))
    subject_ids = sorted(set(subject_ids))

    complete = []
    missing = []
    for sid in subject_ids:
        beh_path = os.path.join(data_dir, f"sub{sid}.mat")
        eeg_path = os.path.join(data_dir, f"sub{sid}_tsk.mat")
        nir_path = os.path.join(data_dir, f"sub{sid}.nirs")
        if os.path.exists(beh_path) and os.path.exists(eeg_path) and os.path.exists(nir_path):
            complete.append(sid)
        else:
            missing.append(sid)

    return complete, missing

complete_subjects, incomplete_subjects = discover_subjects(DATA_DIR)
print(f"Subjects with complete BEH+EEG+NIR triplet: {len(complete_subjects)}")
print(complete_subjects)
if incomplete_subjects:
    print(f"\nSkipping (missing at least one file): {incomplete_subjects}")

Subjects with complete BEH+EEG+NIR triplet: 48
[1, 3, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50]


## 2. Fonctions de traitement par sujet

Ce sont les mêmes fonctions que celles validées sur le sujet 1, laissées inchangées : seule leur entrée change, elles sont maintenant paramétrées par `subject_id` / les chemins de fichiers correspondants, plutôt que d'avoir ces valeurs codées en dur.

In [3]:
def band_power(sig, srate, band):
    freqs, psd = welch(sig, fs=srate, nperseg=min(len(sig), int(srate * 2)))
    df = freqs[1] - freqs[0] if len(freqs) > 1 else 1.0
    mask = (freqs >= band[0]) & (freqs <= band[1])
    return (psd[mask] * df).sum() if mask.any() else np.nan


def spectral_entropy(sig, srate):
    freqs, psd = welch(sig, fs=srate, nperseg=min(len(sig), int(srate * 2)))
    psd_norm = psd / (psd.sum() + 1e-12)
    return scipy_entropy(psd_norm + 1e-12)


def is_special_code(code):
    return (code == 251) or (code == 252) or (80 <= code <= 89)


def load_beh(subject_id):
    beh_raw = mat73.loadmat(os.path.join(DATA_DIR, f"sub{subject_id}.mat"))['BEH']
    beh = {k: np.array(beh_raw[k]) for k in ['Acc', 'Rt1', 'Soa', 'Tag']}
    if beh['Soa'].shape != (N_TRIALS_PER_BLOCK, N_BLOCKS):
        raise ValueError(f"Unexpected BEH shape {beh['Soa'].shape}")
    return beh


def load_eeg_blocks(subject_id):
    path = os.path.join(DATA_DIR, f"sub{subject_id}_tsk.mat")
    f_eeg = h5py.File(path, 'r')
    eeg_grp = f_eeg['EEG_TSK']

    def deref(field, block):
        ref = eeg_grp[field][block, 0]
        return f_eeg[ref]

    blocks = []
    for b in range(N_BLOCKS):
        srate = float(deref('srate', b)[()].flatten()[0])
        data = deref('data', b)[()]  # (pnts, nbchan)
        event_grp = deref('event', b)
        ev_types = np.array([f_eeg[r][()].flatten()[0] for r in event_grp['type'][:, 0]])
        ev_lats = np.array([f_eeg[r][()].flatten()[0] for r in event_grp['latency'][:, 0]])

        resp_mask = np.array([80 <= c <= 89 for c in ev_types])
        resp_lats = ev_lats[resp_mask]

        blocks.append(dict(srate=srate, data=data, resp_lats=resp_lats))
    f_eeg.close()
    return blocks


def load_nir(subject_id):
    path = os.path.join(DATA_DIR, f"sub{subject_id}.nirs")
    with h5py.File(path, 'r') as f_nir:
        d = f_nir['d'][:]
        if d.shape[0] != 88 and d.shape[1] == 88:
            d = d.T
        ml = f_nir['ml'][:]
        if ml.shape[0] != 4:
            ml = ml.T
        aux = f_nir['aux'][:]
        if aux.shape[0] != 7 and aux.shape[1] == 7:
            aux = aux.T
        t_nir = f_nir['t'][:].flatten()
        lam = f_nir['SD']['Lambda'][:].flatten()
    return d, ml, aux, t_nir, lam


def nir_to_hbo_hbr(d, ml, lam):
    wl1, wl2 = lam[0], lam[1]
    wl_idx = ml[3, :].astype(int)
    ch_wl1 = np.where(wl_idx == 1)[0]
    ch_wl2 = np.where(wl_idx == 2)[0]
    n_pairs = min(len(ch_wl1), len(ch_wl2))

    # 'd' is already (delta) optical density, not raw intensity -- confirmed on
    # subject 1 (range ~-5..+5, includes negatives). Fed directly into mBLL.
    od = d
    E = np.array([[EXT_COEFFS[wl1]['HbO'], EXT_COEFFS[wl1]['HbR']],
                  [EXT_COEFFS[wl2]['HbO'], EXT_COEFFS[wl2]['HbR']]])
    E_inv = np.linalg.inv(E)

    hbo_all, hbr_all = [], []
    for p in range(n_pairs):
        od1 = od[ch_wl1[p], :]
        od2 = od[ch_wl2[p], :]
        conc = E_inv @ np.vstack([od1, od2]) / (DPF * L)
        hbo_all.append(conc[0])
        hbr_all.append(conc[1])
    return np.array(hbo_all), np.array(hbr_all)


def rising_edges(sig, thresh_frac=0.5):
    thresh = thresh_frac * (sig.max() - sig.min()) + sig.min()
    dd = np.diff((sig > thresh).astype(int))
    return np.where(dd == 1)[0] + 1


def nir_task_windows(aux, t_nir):
    ch6_idx = rising_edges(aux[6])
    rest_times = t_nir[ch6_idx]
    if len(rest_times) != 12:
        raise ValueError(f"Expected 12 rest-boundary pulses, got {len(rest_times)}")
    windows = []
    for i in range(5):
        windows.append((rest_times[2 * i + 1], rest_times[2 * i + 2]))
    windows.append((rest_times[11], t_nir[-1]))
    return windows

## 3. Contrôle qualité des données — une vérification par sujet, avant de faire confiance aux résultats

On applique à chaque sujet les mêmes contrôles que ceux réalisés à l'origine sur le sujet 1 pour le Niveau 1 (nombre de canaux, fréquences d'échantillonnage, nombre de valeurs manquantes (NaN), nombre d'événements comparé à l'attendu (~172 par bloc), et vérification que le motif d'exclusion identifié sur le sujet 1 — les lignes {0, 43} de chaque bloc — se retrouve bien chez ce sujet également). Les résultats de chaque sujet sont ensuite rassemblés dans un seul tableau récapitulatif. Un sujet peut passer le contrôle de présence des fichiers (les 3 fichiers existent) tout en ayant des données de mauvaise qualité à l'intérieur — c'est précisément ce que ce contrôle permet de détecter.

In [4]:
def qc_subject(subject_id):
    flags = []
    result = dict(subject_id=subject_id)

    # --- BEH ---
    beh = load_beh(subject_id)
    result['beh_shape_ok'] = (beh['Soa'].shape == (N_TRIALS_PER_BLOCK, N_BLOCKS))
    if not result['beh_shape_ok']:
        flags.append(f"BEH shape unexpected: {beh['Soa'].shape}")

    soa = beh['Soa']
    outlier_rows, outlier_cols = np.where(soa > 10)
    result['n_soa_outliers'] = len(outlier_rows)
    outlier_row_set = set(outlier_rows.tolist())
    result['soa_outlier_rows'] = sorted(outlier_row_set)
    if outlier_row_set != set(EXCLUDE_TRIAL_ROWS):
        flags.append(f"SOA outlier rows differ from standing rule {EXCLUDE_TRIAL_ROWS}: "
                      f"found {sorted(outlier_row_set)}")

    result['beh_nan_count'] = sum(np.isnan(v).sum() for v in beh.values())

    # --- EEG ---
    eeg_blocks = load_eeg_blocks(subject_id)
    srates = [b['srate'] for b in eeg_blocks]
    result['eeg_srate_consistent'] = len(set(srates)) == 1
    result['eeg_srate'] = srates[0] if result['eeg_srate_consistent'] else srates
    if not result['eeg_srate_consistent']:
        flags.append(f"Inconsistent sampling rate across blocks: {srates}")

    n_resp_per_block = [len(b['resp_lats']) for b in eeg_blocks]
    result['eeg_resp_events_per_block'] = n_resp_per_block
    if any(abs(n - N_TRIALS_PER_BLOCK) > 5 for n in n_resp_per_block):
        flags.append(f"Response-event count far from expected {N_TRIALS_PER_BLOCK}: {n_resp_per_block}")

    eeg_nan_total = sum(np.isnan(b['data']).sum() for b in eeg_blocks)
    result['eeg_nan_count'] = int(eeg_nan_total)
    if eeg_nan_total > 0:
        flags.append(f"{eeg_nan_total} NaN samples in EEG data")

    # --- NIR ---
    d, ml, aux, t_nir, lam = load_nir(subject_id)
    result['nir_channels'] = d.shape[0]
    result['nir_nan_count'] = int(np.isnan(d).sum())
    if result['nir_nan_count'] > 0:
        flags.append(f"{result['nir_nan_count']} NaN samples in NIR data")
    try:
        _ = nir_task_windows(aux, t_nir)
        result['nir_rest_boundaries_ok'] = True
    except Exception as e:
        result['nir_rest_boundaries_ok'] = False
        flags.append(f"NIR rest-boundary segmentation failed: {e}")

    result['flags'] = "; ".join(flags) if flags else "OK"
    result['n_flags'] = len(flags)
    return result


qc_rows = []
for sid in complete_subjects:
    try:
        qc_rows.append(qc_subject(sid))
    except Exception as e:
        qc_rows.append(dict(subject_id=sid, flags=f"QC ITSELF FAILED: {e}", n_flags=-1))

qc_df = pd.DataFrame(qc_rows)
print(f"QC'd {len(qc_df)} subjects. Subjects with at least one flag:")
qc_df[qc_df['n_flags'] != 0][['subject_id', 'flags']]

QC'd 48 subjects. Subjects with at least one flag:


,subject_id,flags
3,5,"SOA outlier rows differ from standing rule [0,..."
4,6,"SOA outlier rows differ from standing rule [0,..."
7,10,"SOA outlier rows differ from standing rule [0,..."
11,14,"SOA outlier rows differ from standing rule [0,..."
14,17,"SOA outlier rows differ from standing rule [0,..."
15,18,"SOA outlier rows differ from standing rule [0,..."
17,20,"SOA outlier rows differ from standing rule [0,..."
21,24,"SOA outlier rows differ from standing rule [0,..."
24,27,"SOA outlier rows differ from standing rule [0,..."
27,30,"SOA outlier rows differ from standing rule [0,..."


In [5]:
# Full QC table -- review before trusting extraction results below
qc_df

,subject_id,beh_shape_ok,n_soa_outliers,soa_outlier_rows,beh_nan_count,eeg_srate_consistent,eeg_srate,eeg_resp_events_per_block,eeg_nan_count,nir_channels,nir_nan_count,nir_rest_boundaries_ok,flags,n_flags
0,1,True,11,"[0, 43]",0,True,500.0,"[172, 172, 172, 172, 172, 172]",0,88,0,True,OK,0
1,3,True,6,"[0, 43]",0,True,500.0,"[172, 172, 172, 172, 172, 172]",0,88,0,True,OK,0
2,4,True,10,"[0, 43]",0,True,500.0,"[172, 172, 172, 172, 172, 172]",0,88,0,True,OK,0
3,5,True,5,[0],0,True,500.0,"[172, 172, 172, 172, 172, 172]",0,88,0,True,"SOA outlier rows differ from standing rule [0,...",1
4,6,True,5,[0],0,True,500.0,"[172, 172, 172, 172, 172, 172]",0,88,0,True,"SOA outlier rows differ from standing rule [0,...",1
5,7,True,6,"[0, 43]",0,True,500.0,"[172, 172, 172, 172, 172, 172]",0,88,0,True,OK,0
6,9,True,6,"[0, 43]",0,True,500.0,"[172, 172, 172, 172, 172, 172]",0,88,0,True,OK,0
7,10,True,5,[0],0,True,500.0,"[172, 172, 172, 172, 172, 172]",0,88,0,True,"SOA outlier rows differ from standing rule [0,...",1
8,11,True,6,"[0, 43]",0,True,500.0,"[172, 172, 172, 172, 172, 172]",0,88,0,True,OK,0
9,12,True,6,"[0, 43]",0,True,500.0,"[172, 172, 172, 172, 172, 172]",0,88,0,True,OK,0


Les sujets marqués `QC ITSELF FAILED` ou présentant un motif d'exclusion SOA inhabituel méritent d'être examinés individuellement avant d'être inclus dans l'analyse. L'étape d'extraction ci-dessous les traitera tout de même (en appliquant systématiquement la règle d'exclusion standard `{0, 43}`), mais leurs résultats doivent être interprétés avec davantage de prudence que ceux des sujets validés sans réserve.


## 4. Pipeline complet par sujet

Combine les données BEH + EEG + NIR dans le même format de tableau de caractéristiques que celui utilisé pour le sujet 1 ; renvoie `None` (et affiche l'erreur) si quoi que ce soit échoue pour ce sujet, afin que la boucle de traitement par lot puisse l'ignorer et continuer.

In [6]:
def process_subject(subject_id):
    valid_mask = np.ones(N_TRIALS_PER_BLOCK, dtype=bool)
    valid_mask[EXCLUDE_TRIAL_ROWS] = False
    valid_rows = np.where(valid_mask)[0]

    beh = load_beh(subject_id)
    eeg_blocks = load_eeg_blocks(subject_id)
    d, ml, aux, t_nir, lam = load_nir(subject_id)
    hbo_all, hbr_all = nir_to_hbo_hbr(d, ml, lam)
    task_windows = nir_task_windows(aux, t_nir)

    frontal_idx = [CHAN_NAMES.index(c) for c in FRONTAL_CHANS if c in CHAN_NAMES]

    rows = []
    for b in range(N_BLOCKS):
        srate = eeg_blocks[b]['srate']
        data = eeg_blocks[b]['data']
        resp_lats = eeg_blocks[b]['resp_lats']
        n_resp = len(resp_lats)
        block_start_nir_t, block_end_nir_t = task_windows[b]

        for row in valid_rows:
            if row >= n_resp:
                continue
            resp_lat = resp_lats[row]
            rt1 = beh['Rt1'][row, b]
            if np.isnan(resp_lat) or np.isnan(rt1):
                continue

            stim_sample = int(resp_lat - rt1 * srate)
            resp_sample = int(resp_lat)

            pre_start = max(0, stim_sample - int(1.5 * srate))
            pre_end = stim_sample
            if pre_end <= pre_start or pre_end > data.shape[0]:
                continue
            pre_window = data[pre_start:pre_end, :]

            theta = np.nanmean([band_power(pre_window[:, c], srate, BAND_THETA)
                                 for c in range(pre_window.shape[1]) if c < 32])
            alpha = np.nanmean([band_power(pre_window[:, c], srate, BAND_ALPHA)
                                 for c in range(pre_window.shape[1]) if c < 32])
            beta = np.nanmean([band_power(pre_window[:, c], srate, BAND_BETA)
                                for c in range(pre_window.shape[1]) if c < 32])
            ab_ratio = alpha / beta if beta > 0 else np.nan
            sent = np.nanmean([spectral_entropy(pre_window[:, c], srate)
                                for c in range(pre_window.shape[1]) if c < 32])

            frn_start = max(0, resp_sample - int(0.1 * srate))
            frn_end = min(data.shape[0], resp_sample + int(0.4 * srate))
            base_start = max(0, resp_sample - int(0.1 * srate))
            base_end = resp_sample
            if frn_end > frn_start and base_end > base_start:
                frn = data[frn_start:frn_end, frontal_idx].mean() - data[base_start:base_end, frontal_idx].mean()
            else:
                frn = np.nan

            stim_time_in_block = (resp_lat - rt1 * srate) / srate
            abs_stim_time = block_start_nir_t + stim_time_in_block
            idx_start = np.searchsorted(t_nir, abs_stim_time)
            idx_end = np.searchsorted(t_nir, abs_stim_time + 10.0)
            if idx_end <= idx_start or idx_end > hbo_all.shape[1]:
                continue
            mean_hbo = hbo_all[:, idx_start:idx_end].mean()
            mean_hbr = hbr_all[:, idx_start:idx_end].mean()

            rows.append(dict(
                subject_id=subject_id, block=b, trial_row=int(row),
                theta=theta, alpha=alpha, beta=beta,
                alpha_beta_ratio=ab_ratio, spectral_entropy=sent, frn=frn,
                mean_hbo=mean_hbo, mean_hbr=mean_hbr,
                soa=beh['Soa'][row, b], acc=beh['Acc'][row, b], tag=beh['Tag'][row, b],
                rt1=rt1,
            ))

    df = pd.DataFrame(rows)
    df['uncertainty_proxy'] = (1 - df['acc']) * 0.6 + (df['soa'] / df['soa'].max()) * 0.4
    return df

## 5. Exécution du traitement par lot

In [7]:
all_dfs = []
failed_subjects = {}

for sid in complete_subjects:
    try:
        sdf = process_subject(sid)
        if len(sdf) == 0:
            raise ValueError("0 trials extracted")
        all_dfs.append(sdf)
        print(f"sub{sid}: {len(sdf)} trials OK")
    except Exception as e:
        failed_subjects[sid] = str(e)
        print(f"sub{sid}: FAILED - {e}")

print(f"\n{len(all_dfs)} / {len(complete_subjects)} subjects processed successfully")
if failed_subjects:
    print("Failed subjects:", failed_subjects)

sub1: 1012 trials OK
sub3: 1018 trials OK
sub4: 1019 trials OK
sub5: 1008 trials OK
sub6: 1019 trials OK
sub7: 1016 trials OK
sub9: 1019 trials OK
sub10: 1012 trials OK
sub11: 1020 trials OK
sub12: 1014 trials OK
sub13: 980 trials OK
sub14: 1020 trials OK
sub15: 1020 trials OK
sub16: 1020 trials OK
sub17: 1013 trials OK
sub18: 1020 trials OK
sub19: 1020 trials OK
sub20: 1020 trials OK
sub21: 1016 trials OK
sub22: 1012 trials OK
sub23: 1004 trials OK
sub24: 1018 trials OK
sub25: 1020 trials OK
sub26: 1016 trials OK
sub27: 989 trials OK
sub28: 1019 trials OK
sub29: 910 trials OK
sub30: 977 trials OK
sub31: 1010 trials OK
sub32: 1010 trials OK
sub33: 1020 trials OK
sub34: 1018 trials OK
sub35: 1007 trials OK
sub36: 1020 trials OK
sub37: 1020 trials OK
sub38: 1016 trials OK
sub39: 991 trials OK
sub40: 1018 trials OK
sub41: 1020 trials OK
sub42: 1012 trials OK
sub43: 1019 trials OK
sub44: 1016 trials OK
sub45: 1020 trials OK
sub46: 919 trials OK
sub47: 1014 trials OK
sub48: 1018 trials OK
s

In [8]:
combined = pd.concat(all_dfs, ignore_index=True)
print(f"Combined table: {len(combined)} trials x {combined.shape[1]} columns, "
      f"from {combined['subject_id'].nunique()} subjects")

combined.to_csv(os.path.join(DATA_DIR, "level2_features_all_subjects.csv"), index=False)
print(f"Saved to {os.path.join(DATA_DIR, 'level2_features_all_subjects.csv')}")
combined.head()

Combined table: 48397 trials x 16 columns, from 48 subjects
Saved to data\level2_features_all_subjects.csv


,subject_id,block,trial_row,theta,alpha,beta,alpha_beta_ratio,spectral_entropy,frn,mean_hbo,mean_hbr,soa,acc,tag,rt1,uncertainty_proxy
0,1,0,1,4.007698,27.537148,37.084627,0.742549,3.189449,-7.694791,0.000017,0.000030,4.637086,1.0,1.0,0.333122,0.372357
1,1,0,2,6.007094,27.728138,31.292710,0.886089,2.887930,-13.304374,0.000018,0.000029,3.396587,1.0,1.0,0.254548,0.272745
2,1,0,3,6.380240,29.946643,45.442523,0.659000,3.311367,-0.496750,0.000020,0.000028,3.860776,1.0,1.0,0.334574,0.310020
3,1,0,4,16.769313,32.066125,27.549029,1.163966,3.167648,-13.857596,0.000020,0.000029,4.258895,1.0,1.0,0.280883,0.341989
4,1,0,5,3.073111,16.149156,29.871550,0.540620,3.236856,-10.300018,0.000018,0.000029,4.443033,1.0,1.0,0.272091,0.356775


## 7. Fonction de risque (hazard) empirique du SOA — référence théorique, pas une cible d'apprentissage

Cette fonction est calculée une seule fois, à partir de la distribution regroupée des valeurs valides de SOA sur l'ensemble des sujets (méthode actuarielle / table de survie, l'estimateur non paramétrique standard du hazard utilisé dans la littérature sur le foreperiod — Niemi & Näätänen). Elle n'est **jamais** utilisée comme cible d'entraînement : `hazard(SOA)` est une fonction déterministe du SOA, et le SOA est déjà une caractéristique d'entrée du modèle. Si on l'utilisait comme cible de régression, un modèle pourrait « gagner » en ignorant complètement l'EEG et le fNIRS, en se contentant d'apprendre cette relation mathématique fixe — un raccourci sans aucune valeur scientifique.

`hazard` est donc simplement ajoutée ici comme colonne, pour que le Niveau 3 puisse l'utiliser uniquement comme **référence théorique a posteriori** : un modèle qui prédit le temps de réaction (RT) à partir de `hazard(SOA)` seul (sans aucune donnée cérébrale) fixe le niveau plancher « expliqué par le design expérimental seul ». Le modèle fusionné EEG+fNIRS+SOA doit dépasser ce plancher pour démontrer que les marqueurs neuronaux apportent réellement une information supplémentaire.

In [9]:
def compute_hazard_function(soa_values, n_bins=15):
    # Actuarial/life-table hazard estimate -- see markdown above.
    soa_values = np.asarray(soa_values)
    soa_values = soa_values[~np.isnan(soa_values)]
    bins = np.linspace(soa_values.min(), soa_values.max(), n_bins + 1)
    hazard_lookup = np.zeros(n_bins)
    for i in range(n_bins):
        bin_start, bin_end = bins[i], bins[i + 1]
        if i < n_bins - 1:
            events_in_bin = np.sum((soa_values >= bin_start) & (soa_values < bin_end))
        else:
            events_in_bin = np.sum((soa_values >= bin_start) & (soa_values <= bin_end))
        at_risk = np.sum(soa_values >= bin_start)  # "survived" to this elapsed time without target yet
        hazard_lookup[i] = events_in_bin / at_risk if at_risk > 0 else np.nan
    return bins, hazard_lookup


def hazard_for_soa(soa, bins, hazard_lookup):
    idx = np.digitize(soa, bins) - 1
    idx = np.clip(idx, 0, len(hazard_lookup) - 1)
    return hazard_lookup[idx]


valid_soa_pool = combined['soa'].to_numpy()
hazard_bins, hazard_lookup = compute_hazard_function(valid_soa_pool, n_bins=15)

print("Hazard function (elapsed time bin -> conditional probability target appears now):")
for i in range(len(hazard_lookup)):
    print(f"  [{hazard_bins[i]:.2f}-{hazard_bins[i+1]:.2f}s]: hazard = {hazard_lookup[i]:.3f}")

combined['hazard'] = hazard_for_soa(combined['soa'].to_numpy(), hazard_bins, hazard_lookup)

# Sanity check: classic foreperiod effect -- higher hazard (more time elapsed, more
# 'due' to appear) should correlate with FASTER (lower) real RT, if this dataset
# shows the expected effect at all.
corr = combined[['hazard', 'rt1']].corr().iloc[0, 1]
print(f"\nCorrelation(hazard, real RT1) across all trials: {corr:.3f}  "
      f"(expected: negative -- higher hazard -> faster RT)")

combined.to_csv(os.path.join(DATA_DIR, "level2_features_all_subjects.csv"), index=False)
print("Re-saved with 'hazard' column added.")

Hazard function (elapsed time bin -> conditional probability target appears now):
  [1.97-2.17s]: hazard = 0.001
  [2.17-2.37s]: hazard = 0.010
  [2.37-2.57s]: hazard = 0.018
  [2.57-2.78s]: hazard = 0.030
  [2.78-2.98s]: hazard = 0.046
  [2.98-3.18s]: hazard = 0.054
  [3.18-3.38s]: hazard = 0.066
  [3.38-3.58s]: hazard = 0.085
  [3.58-3.79s]: hazard = 0.112
  [3.79-3.99s]: hazard = 0.135
  [3.99-4.19s]: hazard = 0.187
  [4.19-4.39s]: hazard = 0.242
  [4.39-4.60s]: hazard = 0.350
  [4.60-4.80s]: hazard = 0.587
  [4.80-5.00s]: hazard = 1.000
\nCorrelation(hazard, real RT1) across all trials: -0.002  (expected: negative -- higher hazard -> faster RT)
Re-saved with 'hazard' column added.


## 8. Vérification de cohérence par sujet

Un aperçu rapide du nombre d'essais et des valeurs manquantes pour chaque sujet : un sujet avec nettement moins d'essais que les autres, ou avec beaucoup de valeurs manquantes (NaN), mérite d'être revérifié avant d'être intégré à l'ensemble utilisé pour le Niveau 3.

In [10]:
summary = combined.groupby('subject_id').agg(
    n_trials=('theta', 'size'),
    n_missing_theta=('theta', lambda s: s.isna().sum()),
    n_missing_hbo=('mean_hbo', lambda s: s.isna().sum()),
).reset_index()
summary

,subject_id,n_trials,n_missing_theta,n_missing_hbo
0,1,1012,0,0
1,3,1018,0,0
2,4,1019,0,0
3,5,1008,0,0
4,6,1019,0,0
5,7,1016,0,0
6,9,1019,0,0
7,10,1012,0,0
8,11,1020,0,0
9,12,1014,0,0
